In [138]:
from pathlib import Path


def read_graph(filepath):
    filepath = Path(filepath)

    n = None
    source = None
    sink = None
    edges = []

    with filepath.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("c"):
                continue

            parts = line.split()

            if parts[0] == "p":
                n = int(parts[2])

            elif parts[0] == "n":
                node = int(parts[1]) - 1
                if parts[2] == "s":
                    source = node
                elif parts[2] == "t":
                    sink = node

            elif parts[0] == "a":
                u = int(parts[1]) - 1
                v = int(parts[2]) - 1
                cap = int(parts[3])
                edges.append((u, v, cap))

    # пропускная способность
    capacity = [[0] * n for _ in range(n)]
    for u, v, cap in edges:
        capacity[u][v] += cap
    return capacity, source, sink

In [139]:
from collections import deque


def bfs(capacity: list, source: int, sink: int):
    n = len(capacity)
    parent = [-1] * n
    parent[source] = source

    q = deque([source])
    while q:
        v = q.popleft()
        for u in range(n):
            if parent[u] == -1 and capacity[v][u] > 0:
                parent[u] = v
                if u == sink:
                    return parent
                q.append(u)
    return None

In [140]:
import time

INF = 10**18

def edmonds_karp(capacity: list, source: int, sink: int):
    flow = 0
    iterations = 0

    while parent := bfs(capacity, source, sink):
        iterations += 1
        # find max flow for the shortest path
        cur = sink
        mn_flow = INF
        while cur != source:
            prev = parent[cur]
            mn_flow = min(mn_flow, capacity[prev][cur])
            cur = prev
        # update path capacity
        cur = sink
        while cur != source:
            prev = parent[cur]
            capacity[prev][cur] -= mn_flow
            capacity[cur][prev] += mn_flow
            cur = prev
        flow += mn_flow
    return flow, iterations


def run(filepath: str):
    capacity, source, sink = read_graph(filepath)
    start = time.perf_counter()
    max_flow, iterations = edmonds_karp(capacity, source, sink)
    end = time.perf_counter()

    print(f"File: {Path(filepath).name}")
    print("-" * 40)
    print(f"Max flow: {max_flow}")
    print(f"Iterations: {iterations}")
    print(f"Time: {end - start:.6f} sec")
    print("-" * 40, "\n")

In [141]:
def main():
    inputs = Path("inputs/lab03")
    for filename in inputs.iterdir():
        run(filename)

main()

File: qpbo_problem_1.cleanrescale.bq.max
----------------------------------------
Max flow: 6812
Iterations: 31
Time: 0.001665 sec
---------------------------------------- 

File: qpbo_problem_104.cleanrescale.bq.max
----------------------------------------
Max flow: 0
Iterations: 0
Time: 0.000062 sec
---------------------------------------- 

File: qpbo_problem_142.cleanrescale.bq.max
----------------------------------------
Max flow: 5260
Iterations: 23
Time: 0.004579 sec
---------------------------------------- 

File: qpbo_problem_2.cleanrescale.bq.max
----------------------------------------
Max flow: 9005
Iterations: 30
Time: 0.001438 sec
---------------------------------------- 

File: qpbo_problem_278.cleanrescale.bq.max
----------------------------------------
Max flow: 1506
Iterations: 14
Time: 0.000735 sec
---------------------------------------- 

File: qpbo_problem_295.cleanrescale.bq.max
----------------------------------------
Max flow: 2640
Iterations: 21
Time: 0.000995